# 文書の中から質問の答えを抜き出す（抽出型 QA）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_qa/extractive_qa.ipynb)

NLP クックブック 20-02 ／ 質問応答

文書と質問を与えて、文書中の答えに当たる箇所を抜き出したい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

name = "tsmatz/roberta_qa_japanese"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系の T5Tokenizer は未知語 (大文字の英字など) を文末記号 (</s>) の ID にしてしまう。
# 元の spiece.model から語彙とスコアを読み、Unigram のモデルを組み立て直す
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "spiece.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id())
model = AutoModelForQuestionAnswering.from_pretrained(name).eval()

def answer(question, context):
    enc = tokenizer(question, context, return_tensors="pt", truncation="only_second", max_length=512, return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0]
    with torch.no_grad():
        out = model(**enc)
    is_context = torch.tensor([sid == 1 for sid in enc.sequence_ids(0)])          # 文書側のトークンだけを候補にする
    start = out.start_logits[0].masked_fill(~is_context, -1e9).log_softmax(-1)
    end = out.end_logits[0].masked_fill(~is_context, -1e9).log_softmax(-1)
    scores = start[:, None] + end[None, :]                                        # 全 (開始, 終了) の組のスコア
    scores = scores.triu().masked_fill(torch.ones_like(scores).tril(-1).bool(), -1e9)   # 開始 <= 終了
    scores = scores.masked_fill(torch.ones_like(scores).triu(30).bool(), -1e9)          # 長さ 30 トークンまで
    s, e = divmod(scores.argmax().item(), scores.shape[1])
    return context[offsets[s][0]:offsets[e][1]], round(scores[s, e].exp().item(), 3)

context = ("富士山は、静岡県と山梨県にまたがる活火山である。標高は 3776 メートルで、日本の最高峰である。"
           "2013 年に「富士山-信仰の対象と芸術の源泉」としてユネスコの世界文化遺産に登録された。")
for q in ["富士山の高さは？", "富士山はどこにありますか？", "世界遺産に登録されたのはいつ？", "富士山を最初に登った人は誰？"]:
    print(q, "→", answer(q, context))

富士山の高さは？ → ('3776 メートル', 0.88)
富士山はどこにありますか？ → ('山梨県', 0.312)
世界遺産に登録されたのはいつ？ → ('2013 年に', 0.998)
富士山を最初に登った人は誰？ → ('富士山', 0.184)


- モデルは各トークンについて「答えの開始位置らしさ」「終了位置らしさ」を出力する。その和が最大になる区間を答えとする。
- 文書に答えがない質問（最後の例）にも、それらしい箇所を抜き出してしまう。スコアにしきい値を設けるか、回答不能も学習したモデルを使う。
- 生成型の LLM と違い、答えは必ず原文の一部なので、根拠の箇所が確実に分かる。
- transformers 5 系（5.0〜5.23 で確認）では、このモデルのトークナイザー（T5Tokenizer、SentencePiece の Unigram）が未知語の ID を 2 に固定して作られる。このモデルでは 2 は文末記号 `</s>` なので、語彙にない文字（大文字の英字など）が `</s>` として入力され、答えの位置がずれたり文の区切りと誤解されたりする。上のコードでは元の `spiece.model` から Unigram のモデルを組み立て直し、transformers 4 系と同じ分割に戻している（日本語と英語の 315 文で一致を確認）。上流で直れば不要になる。